# 04 — Full-universe template

Parameterised scaffold. No results. Change `UNIVERSE_RULE` and rebuild the panel; dates and seed still come from YAML.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sp500rl.config import load_config
from sp500rl.seed import set_seed

CFG = load_config(ROOT / "configs" / "default.yaml")
SEED = set_seed(int(CFG["seed"]))
print(f"seed={SEED}")
print("train", CFG["dates"]["train_start"], "→", CFG["dates"]["train_end"])
print("test ", CFG["dates"]["test_start"], "→", CFG["dates"]["test_end"])


In [ ]:
# Parameters (not dates — dates stay in configs/default.yaml)
UNIVERSE_RULE = "full_window"  # sandbox | full_window | top_n
INPUT_CSV = ROOT / CFG["paths"]["raw_dir"] / "synthetic_canonical.csv"
ADAPTER = CFG.get("adapter", "generic")

print("universe rule", UNIVERSE_RULE)
print("seed", SEED)
print("dates", CFG["dates"])


In [ ]:
from sp500rl.data.pipeline import build_panel_from_file
from sp500rl.env.make_env import make_poe

panel = build_panel_from_file(
    INPUT_CSV if INPUT_CSV.exists() else None,
    cfg=CFG,
    adapter=ADAPTER,
    universe=UNIVERSE_RULE,
    use_synthetic=not INPUT_CSV.exists(),
)
print("tickers", sorted(panel["tic"].unique()))
print("shape", panel.shape)

# Train / test env handles — plug in any agent from notebooks 01–03.
train_env = make_poe(panel, CFG, mode="train", return_last_action=False)
test_env = make_poe(panel, CFG, mode="test", return_last_action=False)
print("train episode_length", train_env.episode_length, "n", train_env.portfolio_size)
print("test  episode_length", test_env.episode_length)
print("TODO: attach DRLAgent or SB3 here; do not commit results in this template.")
